In [15]:
"""
Driver: Vlasov-Maxwell test-particle distribution in a time-dependent E field.

Evolves a electron distribution function in the presence of a prescribed
time-dependent electric field and fixed magnetic field, comparing tensor-train (DLR) and full solves.
Reference: http://ammar-hakim.org/sj/je/je32/je32-vlasov-test-ptcl.html
"""

'\nDriver: Vlasov-Maxwell test-particle distribution in a time-dependent E field.\n\nEvolves a electron distribution function in the presence of a prescribed\ntime-dependent electric field and fixed magnetic field, comparing tensor-train (DLR) and full solves.\nReference: http://ammar-hakim.org/sj/je/je32/je32-vlasov-test-ptcl.html\n'

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys, time
sys.path.append('../')

from setup_.paths import save_dir, main_dir
from setup_.configs import *
import setup_.helper as helper_test
import setup_test as test_setup

from basis.basis_spatial import SpatialBasis
from basis.basis_k import FourierBasis
from field import Field, ScalarField
from pde_EM import Maxwell
from pde_vlasovEM import VlasovMaxwell
from vlasov_tests import VlasovTest

init args ['/home/erikaye/miniconda3/envs/quimb_env/lib/python3.11/site-packages/ipykernel_launcher.py', '-f', '/home/erikaye/.local/share/jupyter/runtime/kernel-88e788d4-ab57-457c-95ea-cd8facf2149e.json']


In [2]:
save_figs = False
save_data = False
verbose = False

is_sqrt = False
evolve_ion = False

print('is sqrt', is_sqrt)

if save_data or save_figs:

    if is_sqrt:
        sdir = f'{save_dir}/VM02-test-k-sq/'
        fdir = f'{main_dir}/VM02-test-k-sq/'
    else:
        sdir = f'{save_dir}/VM02-test-k/'
        fdir = f'{main_dir}/VM02-test-k/'

    if not os.path.exists(sdir):
        os.makedirs(sdir, exist_ok=True)

    if not os.path.exists(fdir):
        os.makedirs(fdir, exist_ok=True)

    print('fdir', fdir)
    print('sdir', sdir)

is sqrt False


In [3]:
#######################
## plasma parameters ##
#######################

plasma_config = UnitsConfiguration(eps0=1. / 8.0e-2, mu0=1. / 8.0e-2)
charge = plasma_config.e  # unit of charge
eV = plasma_config.eV  # unit of kB*T

mass_e = 1.0  # mass of electron
mass_i = 1836 * mass_e  # mass of ion
n0_e = 1.0  # electron number density
n0_i = n0_e  # ion number density
T_e = T_i = 1.0
ion_config1 = IonConfiguration(n0=n0_i, mass=mass_i, T=T_i, units_config=plasma_config)
elc_config1 = IonConfiguration(n0=n0_i, mass=mass_e, T=T_e, units_config=plasma_config)
vth_e = elc_config1.vth  # electron thermal speed 1/2 m b^2 n0 = sqrt(5.0e-5)
wp_e = elc_config1.wp    # plasma e' frequency
lamD = elc_config1.lamD  # Debye length
wp_i = ion_config1.wp    # plasma i+ frequency
vth_i = ion_config1.vth  # ion thermal speed

In [4]:
####################
# field definition #
####################

### initial fields
Bz0 = 1.0
Ex0 = 0.9  # 1.0
omega = 0.4567  # 1.0
k = 0.0

def Ex(x, t):
    if k == 0.0:
        if isinstance(x, np.ndarray):
            return Ex0 * np.ones(x.shape) * np.cos(omega * t)
        else:
            return Ex0 * np.cos(omega * t)
    else:
        return Ex0 * np.cos(k * x - omega * t)

##########################
# QTT / grid parameters #
##########################
q = 2    # physical dimension of each tensor core
KX = 0   # number of dimensions in real (x) space
KV = 2   # number of dimensions in velocity (v) space
Lx = 0   # number of tensor cores in x (not used)
Lve = 6  # number of tensor cores in electron velocity space
Lvi = 6  # number of tensor cores in ion velocity space (not used)

## QTT mapping
TN_layout = LayoutType.SEQUENTIAL
v_map_key = 'B'
    
T = 0.5    # total simulation time
dt = 0.0125
vmin, vmax = -12 * vth_e, 12 * vth_e


############################
## initialize Vlasov test ##
############################

VP_test = VlasovTest(Lxs=(Lx,), Lves=(Lve, Lve, Lve), Lvis=(Lvi, Lvi, Lvi), KX=KX, KV=KV, q=q,
                     basis_ves=FourierBasis(), basis_vis=FourierBasis(),
                     ion_config=ion_config1, elc_config=elc_config1,
                     coll_type=None,
                     # DMAX=DMAX, cutoff=cutoff, DMAX_F=DMAX_F, compress_style=comp_style, compress_style_mod=0,
                     dt=dt, # do_adapt_dt=do_adapt_dt, dt_frac=dt_frac, dt=dt, T=T, te_order=te_order,
                     grid_layout=TN_layout, ve_map_key=v_map_key,
                     )

dve = (vmax - vmin) / (2 ** Lve)
VP_test.initialize_axes(x_lims=(0, 2 * np.pi), ve_lims=(-np.pi / dve, np.pi / dve), vi_lims=(-12 * vth_i, 12 * vth_i),
                        ve_map_key=v_map_key, basis_ves=FourierBasis(), basis_vis=FourierBasis(),
                        )

VP_test.initialize_grid()


#############################
## start building PDE system
#############################

## get grid information from Axis objects ##

ax_kvx_e, ax_kvy_e, = VP_test.ve_axes
print('ax map', ax_kvx_e.map, ax_kvy_e.map)
print('dvx', ax_kvx_e.dx, 'dvy', ax_kvy_e.dx)
print('cfl limit', ax_kvx_e.dx / (Bz0 * np.max(ax_kvx_e.xpts)), 'dt', dt)

ve_x_min, ve_x_max = vmin, vmax
ve_y_min, ve_y_max = vmin, vmax
vx_e_vals = np.linspace(vmin, vmax, 2 ** Lve, endpoint=False)
vy_e_vals = np.linspace(vmin, vmax, 2 ** Lve, endpoint=False)

ve_kx_min, ve_kx_max = VP_test.ve_lims[0]
ve_ky_min, ve_ky_max = VP_test.ve_lims[1]
tot_x = 1  # x_max - x_min

coords_x = VP_test.pos_coordsys
coords_ve = VP_test.ve_coordsys
coords_vi = VP_test.vi_coordsys

X, Y, Z = coords_x.coords
VX, VY, VZ = coords_ve.coords

npts_x = 1  # ax_x.npts
npts_vxe = ax_kvx_e.npts
npts_vye = ax_kvy_e.npts

## define MPS grids
grid_i = VP_test.grid_i
grid_e = VP_test.grid_e
grid_X = VP_test.grid_X

x_vals = 0  # ax_x.xpts

kvx_e_vals = ax_kvx_e.xpts
kvy_e_vals = ax_kvy_e.xpts

### define vx, vy, v2 mpo for measurement ###
vx_moment_mpo = grid_e.make_mpo_ndim({ax_kvx_e: ax_kvy_e.get_xmultiply_mpo(x_power=1)})
vy_moment_mpo = grid_e.make_mpo_ndim({ax_kvy_e: ax_kvy_e.get_xmultiply_mpo(x_power=1)})
v2e_mpo = test_setup.get_v2_mpo(grid_e, VP_test.ve_axes)

# plt.figure()
# plt.plot(np.real(init_fe_gtn.get_data().reshape(-1)))
# plt.plot(np.imag(init_fe_gtn.get_data().reshape(-1)))
# plt.show()


---------- test parameters -------
vth_e 1.0 vth_i 0.02333800140046683 c 0.08
wp_e 0.282842712474619 d_e 0.282842712474619 lamD 3.5355339059327378
wp_o 0.0066009836198444955 d_p 12.119405926034494
Resolution:  q 2 Lxs (0,) Lves (6, 6, 6) Lvis (6, 6, 6)
Collision Type: None
Compression: DMAX None, cutoff None, compress style 1
Compression (Force): DMAX None, cutoff None
Time Integration: te order 4, te order EM None
T 100 cfl limit 0.9
Layout: LayoutType.SEQUENTIAL Axes order: None
-------------------------------------
------- initialized axes ---------
VE_AX Axis(6,2,VX0) VX0 <basis.basis_k.FourierBasis object at 0x7687dc161f50> x_min -8.377580409572781 x_max 8.11578102177363 dx 0.26179938779914913
VE_AX Axis(6,2,VY1) VY1 <basis.basis_k.FourierBasis object at 0x7687dc161f50> x_min -8.377580409572781 x_max 8.11578102177363 dx 0.26179938779914913
VI_AX Axis(6,2,VX0) VX0 <basis.basis_k.FourierBasis object at 0x76876a225f90> x_min -0.280056016805602 x_max 0.2713042662804269 dx 0.0087517505

In [5]:
################################################
## theoretical results and error measurements ##
################################################

def get_theory_v(t):
    if omega != 1.:
        z = elc_config1.charge
        vx_exact = Ex0 / (z ** 2 - omega ** 2) * (np.sin(z * t) - z * omega * np.sin(omega * t))
        vy_exact = Ex0 / (z ** 2 - omega ** 2) * (np.cos(z * t) - np.cos(omega * t))
    else:
        z = elc_config1.charge
        vx_exact = z * Ex0 / 2 * (t * np.cos(t) + np.sin(t))
        vy_exact = - z ** 2 * Ex0 / 2 * t * np.sin(t)
        
    return vx_exact, vy_exact

def compute_dist_error(t, fe_field):

    vx_exact, vy_exact = get_theory_v(t)
    fe1_kvx = helper_test.maxwellian_k(kvx_e_vals, vth2=vth_e ** 2, density=1.0, flow=vx_exact, is_sqrt=is_sqrt) * n0_e
    fe1_kvy = helper_test.maxwellian_k(kvy_e_vals, vth2=vth_e ** 2, density=1.0, flow=vy_exact, is_sqrt=is_sqrt)

    theory_fe_gtn = grid_e.make_gtn_from_dicts([{ax_kvx_e: fe1_kvx, ax_kvy_e: fe1_kvy}], data_type=DataType.MPS)

    dist_err = theory_fe_gtn.distance(fe_field.component) / theory_fe_gtn.frobenius_norm()
    if verbose:
        print('elc distribution L2 err', err)
        
    return dist_err

#### additional errors
def fit_maxwellian_error(t, fe_field, px_e, py_e):    

    vx_exact, vy_exact = get_theory_v(t)
    
    def maxwellian_func(xy_grid_point, x0, y0, n0, sig2x, sig2y):
        vx, vy = xy_grid_point
        gx = helper_test.maxwellian(vx, vth2=sig2x, density=1.0, flow=x0)
        gy = helper_test.maxwellian(vy, vth2=sig2y, density=1.0, flow=y0)
        out = gx * gy * n0
        return out


    def jac(xy_grid_point, x0, y0, n0, sig2x, sig2y):
        vx, vy = xy_grid_point

        gx = helper_test.maxwellian(vx, vth2=sig2x, density=1.0, flow=x0)
        gy = helper_test.maxwellian(vy, vth2=sig2y, density=1.0, flow=y0)

        ## df/d(x0) = (x-x0)/b * n0 * maxwellian
        d_x0 = (vx - x0) / sig2x * gx * gy * n0
        d_y0 = (vy - y0) / sig2y * gx * gy * n0

        ## df/d(n0) = maxwellian
        d_n0 = gx * gy

        ## df/d(sig2x) = (x-x0)^2 / 2 / b^2 * maxwellian
        d_sig2x = (vx - x0) ** 2 / 2 / sig2x ** 2 * gx * gy * n0
        d_sig2y = (vy - y0) ** 2 / 2 / sig2y ** 2 * gx * gy * n0

        return np.array([d_x0, d_y0, d_n0, d_sig2x, d_sig2y]).T

    dist_data = fe_field.component.get_data()
    dist_data = ax_kvx_e.basis.get_realspace_1D(dist_data, 0)
    dist_data = ax_kvy_e.basis.get_realspace_1D(dist_data, 1)
    if is_sqrt:
        dist_data = np.conj(dist_data) * dist_data

    mesh_vx, mesh_vy = np.meshgrid(vx_e_vals, vy_e_vals, indexing='ij')
    grid_pts = (mesh_vx.reshape(-1), mesh_vy.reshape(-1))
    # out = maxwellian_func((mesh_vx, mesh_vy), *(px_e, py_e, 1., vth_e ** 2, vth_e ** 2))

    try:
        p_opt, p_cov = scipy.optimize.curve_fit(maxwellian_func, grid_pts, dist_data.reshape(-1),
                                                (px_e, py_e, 1., vth_e ** 2, vth_e ** 2),
                                                jac=jac
                                                )
    except RuntimeError:
        p_opt, p_cov = (np.nan, np.nan), np.nan

    fit_vxs = maxwellian_func((mesh_vx, mesh_vy), px_e, py_e, 1.0, vth_e ** 2, vth_e ** 2)
    fit_opt = maxwellian_func((mesh_vx, mesh_vy), *p_opt)

    if verbose:
        print('opt params', p_opt)
        print('meas params', px_e / p_opt[0], py_e / p_opt[1], tot_x, )
        print('theory params', vx_exact, vy_exact, 1.0, vth_e ** 2, vth_e ** 2)

    shape_err = np.linalg.norm(fit_vxs - dist_data) / np.sqrt(npts_vxe * npts_vye)
    fit_err = np.linalg.norm(fit_opt - dist_data) / np.sqrt(npts_vxe * npts_vye)
    norm_err = np.linalg.norm([p_opt[2] - 1.0])
    drift_err = np.linalg.norm([p_opt[0] - vx_exact, p_opt[1] - vy_exact])
    sigma2_err = np.linalg.norm([p_opt[3] - vth_e ** 2, p_opt[4] - vth_e ** 2])

    dist_err = fit_opt - dist_data
    derr_dvx = np.abs(np.diff(dist_err, axis=0, append=0) / ax_kvx_e.dx)
    derr_dvy = np.abs(np.diff(dist_err, axis=1, append=0) / ax_kvy_e.dx)
    deriv_err = [np.linalg.norm(derr_dvx + derr_dvy) / np.sqrt(npts_vxe * npts_vye)]

    avg_vx = p_opt[0]  # vx_e
    avg_vy = p_opt[1]  # vy_e

    # mass_err = np.abs(norm - norm_fe)

    return (avg_vx, avg_vy), (shape_err, fit_err, norm_err, drift_err, sigma2_err, deriv_err)

In [6]:
########################
#### run simulation ####
########################

def run_advec(te_order, dt, tol=1.0e-14, DMAX=None):
    
    ### update compression parameters
    cutoff = tol ** 2
    VP_test.DMAX = DMAX
    VP_test.cutoff = cutoff
    
    ### update time step size
    VP_test.dt = dt
    
    ### get file naming string ###
    fstr = f'w{omega}_' + VP_test.get_fstr()
    print('filename', fstr)

    ######################################
    ## define compression config object ##
    ######################################
    
    compress_config = CompressionConfiguration()
    compress_config.set_compress_opts(1, max_bond=DMAX, cutoff_mode=CUTOFF_MODE, cutoff=cutoff)
    compress_config.set_compress_opts(2, max_bond=DMAX, cutoff_mode=CUTOFF_MODE, cutoff=cutoff * 0.01)
    
    te_compress_i = compress_config
    te_compress_e = compress_config
    print('te compress fe', te_compress_e)
    print('te compress fi', te_compress_i)
    
    compress_opts = te_compress_e.get_compress_opts(1)

    #########################
    ### initialize fields ###
    #########################
    
    ## ion density:
    init_fi_gtn = grid_i.make_empty_gridTN()
    
    ## elec density
    fe1_kvx = helper_test.maxwellian_k(kvx_e_vals, vth2=vth_e ** 2, density=1.0, flow=0., is_sqrt=is_sqrt) * n0_e
    fe1_kvy = helper_test.maxwellian_k(kvy_e_vals, vth2=vth_e ** 2, density=1.0, flow=0., is_sqrt=is_sqrt)
    init_fe_gtn = grid_e.make_gtn_from_dicts([{ax_kvx_e: fe1_kvx, ax_kvy_e: fe1_kvy}], data_type=DataType.MPS)

    ## E field: 0
    Ex0_gtn = grid_X.make_empty_gridTN()
    Ex0_gtn.data = Ex(x_vals, 0)
    Ex_gtn = grid_X.make_empty_gridTN()
    Ey_gtn = grid_X.make_empty_gridTN()
    Ez_gtn = grid_X.make_empty_gridTN()
    
    ## B field:
    Bz0_gtn = grid_X.make_empty_gridTN()
    Bz0_gtn.data = Bz0
    Bx_gtn = grid_X.make_empty_gridTN()
    By_gtn = grid_X.make_empty_gridTN()
    Bz_gtn = grid_X.make_empty_gridTN()
    
    ## specify boundary conditions
    deriv_v = DerivativeConfiguration(left_bc=BCType.PERIODIC)
    init_fe_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_v, ax_kvy_e: deriv_v, })
    # deriv_x_EM = DerivativeConfiguration(left_bc=BCType.PERIODIC)
    # Ex_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_x_EM, ax_kvy_e: deriv_x_EM})
    # Ey_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_x_EM, ax_kvy_e: deriv_x_EM})
    # Ez_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_x_EM, ax_kvy_e: deriv_x_EM})
    # By_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_x_EM, ax_kvy_e: deriv_x_EM})
    # Bz_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_x_EM, ax_kvy_e: deriv_x_EM})
    # Ex0_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_x_EM, ax_kvy_e: deriv_x_EM})
    # Bz0_gtn.ax_deriv_configs.update({ax_kvx_e: deriv_x_EM, ax_kvy_e: deriv_x_EM})

    #### define fields
    init_fe_field = ScalarField('fe', grid_e, data=init_fe_gtn, is_sqrt=is_sqrt, compress_config=te_compress_e)
    init_fi_field = ScalarField('fi', grid_i, data=init_fi_gtn, is_sqrt=is_sqrt, compress_config=te_compress_i)

    ## treat E as a field
    init_E_field = Field('E', grid_X, data={X: Ex0_gtn, Y: Ey_gtn, Z: Ez_gtn}, is_sqrt=True) #, compress_config=te_compress_E)
    E0_field = None

    ## treat B0 as background field
    init_B_field = Field('B', grid_X, data={X: Bx_gtn, Y: By_gtn, Z: Bz_gtn}, is_sqrt=True) #, compress_config=te_compress_B)
    B0_field = Field('B0', grid_X, data={Z: Bz0_gtn}, is_sqrt=True)
    curlB0_field = Field('B0', grid_X, data={})
    
    tot_V = 1
    norm_fe = init_fe_gtn.norm(is_sqrt=is_sqrt) / tot_V
    print('dx', ax_kvx_e.dx, ax_kvy_e.dx, vmax - vmin)
    print('field norm', norm_fe)


    #######################
    ## initialize tests ###
    #######################
    
    ts = [0]
    nt = 0

    nrg_fe = init_fe_gtn.meas_expec(v2e_mpo, is_sqrt=is_sqrt) * mass_e / 2 / tot_x
    nrg_E = np.array(init_E_field.norms(compIDs=coords_x.coords)) ** 2 / 2 / tot_x
    nrg_B = np.array(init_B_field.norms(compIDs=coords_x.coords)) ** 2 / 2 / tot_x
    if verbose:
        print('tot_x', tot_x)
        print('nrg fe', nrg_fe)
        print('nrg E', nrg_E)
        print('nrg B0', nrg_B)

    if verbose:
        print('init bond dims', init_fe_gtn.max_bond(), init_fi_gtn.max_bond())
    max_bond_fe = [init_fe_gtn.max_bond()]
    max_bond_pre = [init_fe_gtn.max_bond()]

    ## measure moments
    px_e = init_fe_gtn.meas_expec(vx_moment_mpo, integ_axes=grid_e.axes, new_grid=grid_X) / tot_x
    py_e = init_fe_gtn.meas_expec(vx_moment_mpo, integ_axes=grid_e.axes, new_grid=grid_X) / tot_x

    nrg_fe_ts = [nrg_fe]
    avg_vx_ts = [px_e]
    avg_vy_ts = [py_e]
    exact_vx_ts = [0.0]
    exact_vy_ts = [0.0]
    errs_avg = [np.linalg.norm([px_e - 0, py_e - 0])]
    errs_dist = [np.nan]  ## distribution - gaussian centered at theoretical px, py
    errs_shape = [np.nan]  ## distribution - gaussian centered at measured px, py
    ## errors with respect to fitted distribution
    errs_norm = [np.nan]  ## distribution - gaussian centered at measured px, py
    errs_drift = [np.nan]  ## distribution - gaussian centered at measured px, py
    errs_sigma2 = [np.nan]  ## distribution - gaussian centered at measured px, py
    errs_fitted = [np.nan]  ## distribution - fitted gaussian
    errs_deriv = [np.nan]  ## (d/dx + d/dy) distribution - fitted gaussian; attempt to measure (lack of) smoothness
    errs_mass = [np.nan]

    ### BUILD PDE
    em_sys = Maxwell(init_E_field, init_B_field,  # phi=init_phi_field, psi=init_psi_field,
                     coords_x=coords_x, matl_params=plasma_config,
                     background_B0=B0_field, background_E0=E0_field,
                     normalize=False, clean=False, is_yee=False)
    em_sys.curlB0 = curlB0_field
    
    vm_sys = VlasovMaxwell(init_fe_field, init_fi_field, em_sys,
                           coords_x=coords_x, coords_ve=coords_ve, coords_vi=coords_vi,
                           elc_params=elc_config1, ion_params=ion_config1, evolve_ion=False, evolve_EM=False,
                           normalize=False,  # True,
                           zipup=True, te_order=te_order,
                           upwind=False, conservative=False,
                           )
    print('initialized vm_sys')
    
    vm_sys.time = ts[-1]
        
    print('E nrg', vm_sys.E.norm() ** 2 * 1. / 2 / tot_x)
    
    px_e = vm_sys.sys_fe.compute_moment([ax_kvx_e], powers=[1], integ_axes=grid_e.axes, compress=0).component / tot_x
    py_e = vm_sys.sys_fe.compute_moment([ax_kvy_e], powers=[1], integ_axes=grid_e.axes, compress=0).component / tot_x
    print('init avg v', px_e, py_e)

    ### plot initial fe
    # fig1, ax1 = plt.subplots(1, 1)
    # time_ = nt * dt
    # fe_data = init_fe_gtn.get_data()
    # fe_data = ax_kvy_e.basis.get_realspace_1D(fe_data, 1)
    # fe_data = ax_kvx_e.basis.get_realspace_1D(fe_data, 0)
    # im1 = ax1.imshow(np.real(fe_data.T), extent=(ve_x_min, ve_x_max, ve_y_min, ve_y_max),
    #                  aspect='auto', origin='lower')
    # fig1.colorbar(im1, ax=ax1)
    # ax1.set_title(f'time {time_:3.2f}')
    # ax1.set_xlabel('vx'), ax1.set_ylabel('vy')
    # plt.show()

    ### plot force exerted onto electrons
    # em_term = vm_sys.compute_force_term(is_ion=False)
    # force_data = em_term.get_comp_data(X)
    # force_data = ax_kvy_e.basis.get_realspace_1D(force_data, 1)
    # force_data = ax_kvx_e.basis.get_realspace_1D(force_data, 0)
    # plt.figure()
    # plt.imshow(np.real(force_data))
    # plt.title('force data')
    # plt.colorbar()
    # plt.show()
    
    time1 = time.time()
    
    print('starting time evolution')
    while ts[-1] < T:
        
        dt_ = (dt * 0.1) if nt == 0 else dt
    
        ## manually update E
        new_Ex = grid_X.make_empty_gridTN()
        if 60 <= te_order < 70 or te_order in [80, 85, 95] or 90 <= te_order < 95:
            new_Ex.data = Ex(x_vals, ts[-1] + dt_ / 2)
            vm_sys.time = ts[-1] + dt_ / 2
            vm_sys.sys_fe.time = ts[-1] + dt_ / 2
            if verbose:
                print('add dt/2 to time', vm_sys.time, vm_sys.sys_fe.time, ts[-1], dt_ / 2)
        else:
            new_Ex.data = Ex(x_vals, ts[-1])  # + dt_/2)
        new_Ex.is_constant = (k == 0.)
        vm_sys.E[X] = new_Ex
    
        walltime = time.time()
        vm_sys = vm_sys.next_time_step(dt_, compress_level=1, err_tol=1.0e-7, verbose_plot=False,
                                       is_first_time_step=(nt == 0),
                                       direction=1)
        if verbose:
            print('wall time', time.time() - walltime)
    
        max_bond_pre += [vm_sys.fe.max_bond()] # [vm_sys.fe.component.info.get('internal_rank', np.nan)]
        max_bond_fe += [vm_sys.fe.max_bond()]
        if verbose:
            print('internal (saved) max bond', max_bond_pre[-1])
            print('post compression max bond', max_bond_fe[-1])        

        norm = vm_sys.fe.norm()  # - vm_sys.fi.norm()
        if is_sqrt:
            norm = np.abs(norm) ** 2
    
        if verbose:
            print('vm_sys.fe.norm', norm, np.abs(vm_sys.fe.norm() - norm_fe),
                  np.abs(np.real(vm_sys.fe.norm()) - norm_fe), np.abs(np.abs(vm_sys.fe.norm()) - norm_fe))
        
        fe_field: 'ScalarField' = vm_sys.fe
        nrg_fe = fe_field.component.meas_expec(v2e_mpo, is_sqrt=is_sqrt) * mass_e / 2 / tot_x
        energy_E_t = np.array(vm_sys.E.norms(compIDs=coords_x.coords)) ** 2 * 1. / 2 / tot_x  # electric field energy
    
        px_e = vm_sys.sys_fe.compute_moment([ax_kvx_e], powers=[1], integ_axes=grid_e.axes, compress=0).component / tot_x
        py_e = vm_sys.sys_fe.compute_moment([ax_kvy_e], powers=[1], integ_axes=grid_e.axes, compress=0).component / tot_x
        px_e = np.real(px_e)
        py_e = np.real(py_e)

        if verbose:
            print('px_e', px_e, py_e)
    
        nrg_fe_ts += [nrg_fe]
        avg_vx_ts += [px_e]
        avg_vy_ts += [py_e]
        
        ts += [ts[-1] + dt_]
        nt += 1
    
        if verbose:
            print('result norm', nt, ts[-1], dt, norm, energy_E_t, fe_field.max_bond(), Lx, Lvi, Lve)
    
        ### compute errors
        vx_exact, vy_exact = get_theory_v(ts[-1])
        errs_avg += [np.linalg.norm([px_e - vx_exact, py_e - vy_exact])]

        err = compute_dist_error(ts[-1], fe_field)
        errs_dist += [err]

        (avg_vx, avg_vy), (shape_err, fit_err, norm_err, drift_err, sigma2_err, deriv_err) = fit_maxwellian_error(ts[-1], fe_field, px_e, py_e)

        # avg_vx_ts += [avg_vx]
        # avg_vy_ts += [avg_vy]
        exact_vx_ts += [vx_exact]
        exact_vy_ts += [vy_exact]
    
        errs_shape += [shape_err]
        errs_fitted += [fit_err]
        errs_norm += [norm_err]
        errs_drift += [drift_err]
        errs_sigma2 += [sigma2_err]
        errs_deriv += [deriv_err]

    plt.figure()
    plt.plot(ts, exact_vx_ts, 'k:', lw=3.0)
    plt.plot(ts, exact_vy_ts, 'k--', lw=3.0)
    plt.plot(ts, avg_vx_ts)
    plt.plot(ts, avg_vy_ts)
    
    return (avg_vx_ts, avg_vy_ts), errs_dist, errs_drift, max_bond_fe, max_bond_pre


In [8]:
## 64: PS DLR-G + RK4, 60: PS DLR-G + CN
## 84: AP DLR-G + RK4, 80: PS DLR-G + CN
## 69: PS DLR-X + RK4, 65: PS DLR-X + CN
## 89: AP DLR-X + RK4, 85: PS DLR-X + CN
## 94: PS DLR-P + RK4, 90: PS DLR-P + CN
## 99: AP DLR-P + RK4, 95: AP DLR-P + CN

dts = np.array([0.0125, 0.0125 / 2, 0.0125 / 4])
errs_dt = []
errs_dt_2 = []
for dt in dts:

    ### set time step size ###
    num_tsteps = int(T / dt)
    print('dt', dt, num_tsteps)

    (avg_vx_ts, avg_vy_ts), errs_dist, errs_drift, max_bond_fe, max_bond_pre = run_advec(95, dt, DMAX=None, tol=1.0e-14)

    errs_dt += [errs_dist[-1]] 
    errs_dt_2 += [errs_drift[-1]] 

    # if dt==0.0125:
    #     plt.figure()
    #     plt.plot(max_bond_pre, label='r')
    #     plt.plot(max_bond_fe, ls='--', label='r_in')
    #     plt.title(f'ranks (L={Lve}, dt={dt})')
    #     plt.ylabel('rank')
    #     plt.xlabel('time step')
    #     plt.legend()

plt.figure()
plt.loglog(1/dts, errs_dt, label='L2 error')
plt.loglog(1/dts, errs_dt_2, label='drift error')
plt.loglog(1/dts, dts**2 * errs_dt[0]/dts[0]**2, 'k:')
plt.loglog(1/dts, dts**4 * errs_dt[0]/dts[0]**4, 'k--')
plt.title('error vs dt')
plt.ylabel('error')
plt.xlabel('1/dt')
plt.legend()
plt.show()

dt 0.0125 40
filename w0.4567_Tr1.00_mr1836.0_v_C10_mbNone_TNSFFB_LVE06_LVE06_LVI06_LVI06_dtf0.9_T100.0_te4
te compress fe svd
bonds: {1: None, 2: None}
cutoffs: {1: 1e-28, 2: 9.999999999999999e-31}
cutoff modes: {1: 'rsum2', 2: 'rsum2'}
midpt: False
te compress fi svd
bonds: {1: None, 2: None}
cutoffs: {1: 1e-28, 2: 9.999999999999999e-31}
cutoff modes: {1: 'rsum2', 2: 'rsum2'}
midpt: False
dx 0.26179938779914913 0.26179938779914913 24.0
field norm 1.0000000000000067
initialized vm_sys
E nrg 0.405
init avg v (-9.278157745935844e-15-5.371642766337968e-15j) (-7.75732011050026e-15-5.489254976897391e-15j)
starting time evolution
force advec
calc time deriv 0.0006250000000000001 0.0006250000000000001
old Ex 0.8999999633364454
new Ex 0.8999999633364454
calc time deriv 0.0012500000000000002 0.0012500000000000002
old Ex 0.8999999633364454
new Ex 0.8999998533457845
calc time deriv 0.0012500000000000002 0.0012500000000000002
old Ex 0.8999999633364454
new Ex 0.8999998533457845
calc time deriv 0.0